In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-stage-3-2026")

print("Path to dataset files:", path)

In [ ]:
# Write your code here
# /kaggle/input/q1-stage-3-2026/PlantVillage/test/Potato___Early_blight/002a55fb-7a3d-4a3a-aca8-ce2d5ebc6925___RS_Early.B 8170.JPG


from torch.utils.data import Dataset
from PIL import Image
import glob

class PotatoDiseaseDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.class_labels = {
            "Potato___Early_blight": 0, "Potato___Late_blight": 1, "Potato___healthy": 2,

        }


        self.image_paths = []
        self.labels = []
        for class_name, label in self.class_labels.items():
            class_images = glob.glob(f"{root_dir}/{class_name}/*.JPG")
            self.image_paths.extend(class_images)
            self.labels.extend([label] * len(class_images))

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image_path = self.image_paths[idx]
        label = self.labels[idx]


        image = Image.open(image_path)

        # Apply transformations (if any)
        if self.transform:
            image = self.transform(image)

        return image, label  # Return processed image & label

In [ ]:
image_path = os.path.join(path,"PlantVillage/test/Potato___Early_blight/002a55fb-7a3d-4a3a-aca8-ce2d5ebc6925___RS_Early.B 8170.JPG")
image = Image.open(image_path)
image

In [ ]:
test_path = os.path.join(path, "PlantVillage", "train/Potato___Early_blight/001187a0-57ab-4329-baff-e7246a9edeb0___RS_Early.B 8178.JPG")
#/kaggle/input/q1-stage-3-2026/PlantVillage/test/Potato___healthy/30937333-8898-4634-8c00-af57d3020ba6___RS_HL 1922.JPG

image = Image.open(test_path)
image
#/kaggle/input/q1-stage-3-2026/PlantVillage/test/Potato___Late_blight/0114b195-844c-4978-93a9-b0d5aae010f1___RS_LB 2738.JPG

#/kaggle/input/q1-stage-3-2026/PlantVillage/test/Potato___Early_blight/002a55fb-7a3d-4a3a-aca8-ce2d5ebc6925___RS_Early.B 8170.JPG

In [ ]:

from torchvision import transforms
from torch.utils.data import DataLoader
import os
import numpy as np
import matplotlib.pyplot as plt

# Define transformations
transform = transforms.Compose([
    transforms.Resize((32, 32)),  # Resize images
    transforms.RandomRotation(15),  # Rotate images randomly within ±15 degrees
    transforms.ToTensor(),  # Convert to tensor
      # value for each channel
])

# Validation and testing data typically don’t require augmentations, as we only evaluate the model performance on these sets.
# Instead, we apply basic transformations to prepare the images.
transform_valid_test = transforms.Compose([
    transforms.Resize((32, 32)),  # Resize images to 64x64
    transforms.ToTensor(),  # Convert to tensor

])



# Initialize dataset for Train
train_path = os.path.join(path, "PlantVillage", "train")
test_path = os.path.join(path, "PlantVillage", "test")

classes =  ["Early blight", " Late blight", " Healthy"]

train_dataset = PotatoDiseaseDataset(train_path, transform=transform)
test_dataset = PotatoDiseaseDataset(test_path, transform=transform_valid_test)

# Create DataLoader
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=True, num_workers=2)

# Get a batch of training images
# Show image


fig, axes = plt.subplots(1, 5, figsize=(15, 5))

imgs_indices = [765,99,12,1696,43]

for i in range(5):
    img, label = train_dataset[imgs_indices[i]]
    img_np = img.numpy().transpose(1, 2, 0)

    img_np = np.clip(img_np, 0, 1)
    axes[i].imshow(img_np)
    axes[i].axis('off')
    axes[i].set_title(classes[label])



plt.show()



In [ ]:
# Write your code here

import torch.nn as nn


class MyCNN(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()
        self.features = nn.Sequential(
                  # input 32x32x3
            nn.Conv2d(3, 12, kernel_size=3, padding=1),   #out  32-3+2+1 = 32 x 32 x 12
            # Output: 32 x 32 x 64
            #nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),

            nn.Conv2d(12, 24, kernel_size=3, padding=1),      # 32 - 3 + 2 +1 = 32 x 32 x 24
            # Output: 32 x 32 x 64
            # Two 3x3 convs → receptive field = 5x5
            #nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),

            # Input after pool(enc1): 128 x 128 x 64
            nn.Conv2d(24, 36, kernel_size=5, padding=1),  # 32 - 5 +2 +1 = 30  x 30 x36

            # Output: 16 x 16 x 128
            #nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),

            nn.Conv2d(36, 48, kernel_size=7, padding=1),  #  30 - 7 + 2 +1 = 26 x 26 x 48
            # Output: 16 x 16 x 128
            #nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),

            # Input after pool(enc2): 64 x 64 x 128
            nn.Conv2d(48, 60 , kernel_size=7, padding=1),   # 26 - 7 + 2 + 1 = 22 x 22 x 60
            # Output: 16 x 16 x 256
            #nn.BatchNorm2d(256),
            nn.ReLU(inplace=True)
                                            # [B,64,7,7]
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),                                # [B, 64*7*7]
            nn.Linear( 22 * 22 * 60, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes)                  # logits
        )


    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x







In [ ]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MyCNN().to(device)

model

In [ ]:
# Write your code here


from tqdm import tqdm





def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for images, labels in tqdm(dataloader):
        images, labels = images.to(device), labels.to(device)



        outputs = model(images).squeeze()
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()


        predictions = torch.softmax(outputs, dim=1)
        predictions = torch.argmax(predictions, dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    avg_loss = total_loss / len(dataloader)
    accuracy = 100 * correct / total
    return avg_loss, accuracy


def validate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images).squeeze()
            loss = criterion(outputs, labels)
            total_loss += loss.item()


            predictions = torch.softmax(outputs, dim=1)
            predictions = torch.argmax(predictions,dim=1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    avg_loss = total_loss / len(dataloader)
    accuracy = 100 * correct / total
    return avg_loss, accuracy




In [ ]:
# Write your code here
import torch.optim as optim


criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)
num_epochs = 10



train_losses = []
val_losses = []
train_accuracies = []
val_accuracies = []

for epoch in range(num_epochs):
    train_loss, train_accuracy = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_accuracy = validate(model, test_loader, criterion, device)


    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accuracies.append(train_accuracy)
    val_accuracies.append(val_accuracy)

    print(f"Epoch {epoch+1}/{num_epochs}: "
          f"Train Loss={train_loss:.4f}, Train Accuracy={train_accuracy:.2f}%, "
          f"Val Loss={val_loss:.4f}, Val Accuracy={val_accuracy:.2f}%")


In [ ]:
import matplotlib.pyplot as plt

# Plot loss curve
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(range(1, num_epochs+1), train_losses, label="Train Loss", marker='o')
plt.plot(range(1, num_epochs+1), val_losses, label="Validation Loss", marker='o')
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Loss Curve")
plt.legend()

# Plot accuracy curve
plt.subplot(1, 2, 2)
plt.plot(range(1, num_epochs+1), train_accuracies, label="Train Accuracy", marker='o')
plt.plot(range(1, num_epochs+1), val_accuracies, label="Validation Accuracy", marker='o')
plt.xlabel("Epochs")
plt.ylabel("Accuracy (%)")
plt.title("Accuracy Curve")
plt.legend()

plt.show()

In [ ]:
# Write your code here
